In [91]:
import pandas as pd
from PIL import Image

image = Image.open("train/129.jpg")
w, h = image.size
s1 = w*h

df_train = pd.read_csv("train_data.csv")
s2 = df_train.Label.value_counts()[1] / df_train.Label.value_counts()[0]

subtasks = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": [0, 0],
    "answer": [s1, s2]
})

subtasks

,subtaskID,datapointID,answer
0,1,0,353808.000000
1,2,0,0.715385


In [92]:
import os
import cv2
import numpy as np

IMG_SIZE = 128
X, y = [], []

for _, row in df_train.iterrows():
    img_path = f"train/{row['CodeID']}.jpg"
    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        X.append(img)
        y.append(row['Label'])

X = np.array(X) / 255.0
y= np.array(y)

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [93]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

model = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomContrast(factor=0.10),
    layers.BatchNormalization(),
    layers.Conv2D(filters=128, kernel_size=3, activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPool2D(),
    layers.Conv2D(filters=64, kernel_size=3, activation='relu'),
    layers.MaxPool2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

early_stopping = EarlyStopping(min_delta=0.001, patience=20, restore_best_weights=True)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['binary_accuracy'])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), batch_size=32, epochs=100, callbacks=[early_stopping])

y_preds = model.predict(X_valid)
y_preds = (y_preds > 0.5).astype(int)
accuracy_score(y_preds, y_valid)

Epoch 1/100


C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - binary_accuracy: 0.5547 - loss: 1.0137 - val_binary_accuracy: 0.6000 - val_loss: 0.6443
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 957ms/step - binary_accuracy: 0.6223 - loss: 0.6245 - val_binary_accuracy: 0.6000 - val_loss: 0.6405
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 870ms/step - binary_accuracy: 0.7007 - loss: 0.5831 - val_binary_accuracy: 0.6000 - val_loss: 0.6825
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 899ms/step - binary_accuracy: 0.6832 - loss: 0.6263 - val_binary_accuracy: 0.6000 - val_loss: 0.6561
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 910ms/step - binary_accuracy: 0.6866 - loss: 0.6433 - val_binary_accuracy: 0.6000 - val_loss: 0.6581
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 907ms/step - binary_accuracy: 0.7315 - loss: 0.5825 - val_binary_accuracy: 0.6000 - val_loss: 0.6823
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 887ms/step - binary_accuracy: 0.6807 - loss: 0.5684 - val_binary_accuracy: 0.6000 - val_loss: 0.6476
Epoch 8/100
6/6 ━━━━━━━━━━━━

0.6666666666666666

In [94]:
test_dir = "test"
test_images = []
test_ids = []

for fname in os.listdir(test_dir):
    if fname.endswith(".jpg"):
        img_id = int(fname.split('.')[0])
        img_path = os.path.join(test_dir, fname)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            test_images.append(img)
            test_ids.append(img_id)

X_test = np.array(test_images) / 255.0
preds = model.predict(X_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step


In [95]:
final_preds = (preds > 0.5).astype(int)
final_preds = [pred[0] for pred in final_preds]

df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": test_ids,
    "answer": final_preds
})

submission = pd.concat([subtasks, df_preds])
submission.to_csv("submission.csv", index=False)
submission

,subtaskID,datapointID,answer
0,1,0,353808.000000
1,2,0,0.715385
0,3,0,0.000000
1,3,1,0.000000
2,3,10,0.000000
...,...,...,...
112,3,63,0.000000
113,3,64,0.000000
114,3,7,0.000000
115,3,8,0.000000


In [96]:
submission.answer.value_counts()

answer
0.000000         109
1.000000           8
353808.000000      1
0.715385           1
Name: count, dtype: int64